INSTALL LIBRARY

In [3]:
!pip install pandas

In [4]:
!pip install requests

In [5]:
import json
import pandas as pd
import requests
import sys


import os
import requests
from dotenv import load_dotenv

In [6]:
# CEK LOKASI FILE ENV

from dotenv import find_dotenv
env_path = find_dotenv()
print("Lokasi .env:", env_path)

Lokasi .env: d:\Latihan PPKD\github-miniproject\miniproject-ppkd\miniproject-ppkd\.env


In [7]:
# CEK KONEKSI API
import os
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("COMTRADE_API_KEY")

if API_KEY:
    print("API key berhasil dimuat.")
else:
    print("API key belum ditemukan. Periksa file .env dan nama COMTRADE_API_KEY.")

API key berhasil dimuat.


In [8]:
ALAMAT_API = "https://comtradeapi.un.org/data/v1/get/C/A/HS"

parameter_uji = {
    "reporterCode": "360",
    "partnerCode": "156",
    "flowCode": "M",
    "period": "2024",
    "cmdCode": "AG6",
    "maxRecords": 5,
    "includeDesc": "true",
    "subscription-key": API_KEY,
}

if not API_KEY:
    raise ValueError("API key belum tersedia. Isi COMTRADE_API_KEY pada file .env.")

response_uji = requests.get(ALAMAT_API, params=parameter_uji, timeout=60)
print("Status code:", response_uji.status_code)

response_uji.raise_for_status()
hasil_uji = response_uji.json()
print("Jumlah record diterima:", len(hasil_uji.get("data", [])))

Status code: 200
Jumlah record diterima: 5


In [9]:
# Melihat struktur satu record tanpa menampilkan API key
data_uji = hasil_uji.get("data", [])
if not data_uji:
    raise ValueError("API berhasil dihubungi, tetapi tidak mengembalikan data.")
record_pertama = data_uji[0]
record_pertama

{'typeCode': 'C',
 'freqCode': 'A',
 'refPeriodId': 20240101,
 'refYear': 2024,
 'refMonth': 52,
 'period': '2024',
 'reporterCode': 360,
 'reporterISO': 'IDN',
 'reporterDesc': 'Indonesia',
 'flowCode': 'M',
 'flowDesc': 'Import',
 'partnerCode': 156,
 'partnerISO': 'CHN',
 'partnerDesc': 'China',
 'partner2Code': 0,
 'partner2ISO': 'W00',
 'partner2Desc': 'World',
 'classificationCode': 'H6',
 'classificationSearchCode': 'HS',
 'isOriginalClassification': True,
 'cmdCode': '999999',
 'cmdDesc': 'Commodities not specified according to kind',
 'aggrLevel': 6,
 'isLeaf': True,
 'customsCode': 'C00',
 'customsDesc': 'TOTAL CPC',
 'mosCode': '0',
 'motCode': 0,
 'motDesc': 'TOTAL MOT',
 'qtyUnitCode': -1,
 'qtyUnitAbbr': 'N/A',
 'qty': 0.0,
 'isQtyEstimated': False,
 'altQtyUnitCode': -1,
 'altQtyUnitAbbr': 'N/A',
 'altQty': 0.0,
 'isAltQtyEstimated': False,
 'netWgt': 0.0,
 'isNetWgtEstimated': False,
 'grossWgt': 0.0,
 'isGrossWgtEstimated': False,
 'cifvalue': 201725107.0,
 'fobvalue':

In [10]:
class KlienComtrade:
    """Klien sederhana untuk mengambil impor Indonesia dari China."""

    def __init__(self, api_key, timeout=60, jumlah_retry=2):
        if not api_key:
            raise ValueError("API key wajib diisi.")
        self.api_key = api_key
        self.timeout = timeout
        self.jumlah_retry = jumlah_retry
        self.alamat_api = "https://comtradeapi.un.org/data/v1/get/C/A/HS"

    def ambil_data_impor(self, tahun=2024, maksimum_record=500):
        parameter = {
            "reporterCode": "360",
            "partnerCode": "156",
            "flowCode": "M",
            "period": str(tahun),
            "cmdCode": "AG6",
            "maxRecords": int(maksimum_record),
            "includeDesc": "true",
            "subscription-key": self.api_key,
        }

        error_terakhir = None
        for percobaan in range(1, self.jumlah_retry + 2):
            try:
                response = requests.get(
                    self.alamat_api, params=parameter, timeout=self.timeout
                )
                response.raise_for_status()
                hasil = response.json()

                if "data" not in hasil:
                    raise ValueError("Respons API tidak memiliki key 'data'.")

                return pd.DataFrame(hasil["data"])
            except (requests.RequestException, ValueError) as error:
                error_terakhir = error
                if percobaan <= self.jumlah_retry:
                    print(f"Percobaan {percobaan} gagal. Mengulang...")
                    time.sleep(2 * percobaan)

        raise RuntimeError(f"Pengambilan data gagal: {error_terakhir}")

In [11]:
klien = KlienComtrade(API_KEY)
df_mentah = klien.ambil_data_impor(tahun=2024, maksimum_record=500)

print(f"Jumlah baris mentah: {len(df_mentah)}")
print(f"Jumlah kolom mentah: {len(df_mentah.columns)}")
df_mentah.head()

Jumlah baris mentah: 500
Jumlah kolom mentah: 47


,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,20240101,2024,52,2024,360,IDN,Indonesia,M,...,0.0,False,0.0,False,201725107.0,None,201725107.0,0,False,True
1,C,A,20240101,2024,52,2024,360,IDN,Indonesia,M,...,0.0,False,0.0,False,1547365.0,None,1547365.0,0,True,False
2,C,A,20240101,2024,52,2024,360,IDN,Indonesia,M,...,0.0,False,0.0,False,200177224.0,None,200177224.0,0,True,False
3,C,A,20240101,2024,52,2024,360,IDN,Indonesia,M,...,0.0,False,0.0,False,518.0,None,518.0,0,True,False
4,C,A,20240101,2024,52,2024,360,IDN,Indonesia,M,...,NaN,False,0.0,False,27.0,None,27.0,0,True,False


In [12]:
PETA_KOLOM = {
    "period": "Tahun",
    "reporterCode": "Kode_Reporter",
    "reporterDesc": "Negara_Reporter",
    "partnerCode": "Kode_Mitra",
    "partnerDesc": "Negara_Mitra",
    "flowCode": "Kode_Arus",
    "flowDesc": "Arus_Perdagangan",
    "cmdCode": "Kode_HS",
    "cmdDesc": "Nama_Barang",
    "qtyUnitAbbr": "Satuan_Kuantitas",
    "qty": "Kuantitas",
    "netWgt": "Berat_Bersih_Kg",
    "primaryValue": "Nilai_Impor_USD",
    "isReported": "Dilaporkan_Langsung",
}

kolom_tersedia = [kolom for kolom in PETA_KOLOM if kolom in df_mentah.columns]
df_impor = df_mentah[kolom_tersedia].rename(columns=PETA_KOLOM).copy()
print("Kolom terpilih:", df_impor.columns.tolist())
df_impor.head()

Kolom terpilih: ['Tahun', 'Kode_Reporter', 'Negara_Reporter', 'Kode_Mitra', 'Negara_Mitra', 'Kode_Arus', 'Arus_Perdagangan', 'Kode_HS', 'Nama_Barang', 'Satuan_Kuantitas', 'Kuantitas', 'Berat_Bersih_Kg', 'Nilai_Impor_USD', 'Dilaporkan_Langsung']


,Tahun,Kode_Reporter,Negara_Reporter,Kode_Mitra,Negara_Mitra,Kode_Arus,Arus_Perdagangan,Kode_HS,Nama_Barang,Satuan_Kuantitas,Kuantitas,Berat_Bersih_Kg,Nilai_Impor_USD,Dilaporkan_Langsung
0,2024,360,Indonesia,156,China,M,Import,999999,Commodities not specified according to kind,N/A,0.0,0.0,201725107.0,False
1,2024,360,Indonesia,156,China,M,Import,999999,Commodities not specified according to kind,N/A,0.0,0.0,1547365.0,True
2,2024,360,Indonesia,156,China,M,Import,999999,Commodities not specified according to kind,N/A,0.0,0.0,200177224.0,True
3,2024,360,Indonesia,156,China,M,Import,999999,Commodities not specified according to kind,N/A,0.0,0.0,518.0,True
4,2024,360,Indonesia,156,China,M,Import,711019,"Metals; platinum, semi-manufactured",N/A,0.0,NaN,27.0,True


In [33]:
print("1. Nilai kosong per kolom:")
print(df_impor.isna().sum())
print(f"\n2. Nilai Duplikat berdasarkan Tahun + Kode HS : {df_impor.duplicated(subset=['Tahun', 'Kode_HS']).sum()}")
print("\n3. Tipe data awal:")
print(df_impor.dtypes)

1. Nilai kosong per kolom:
Tahun                  0
Kode_Reporter          0
Negara_Reporter        0
Kode_Mitra             0
Negara_Mitra           0
Kode_Arus              0
Arus_Perdagangan       0
Kode_HS                0
Nama_Barang            0
Satuan_Kuantitas       0
Kuantitas              0
Berat_Bersih_Kg        9
Nilai_Impor_USD        0
Dilaporkan_Langsung    0
dtype: int64

2. Nilai Duplikat berdasarkan Tahun + Kode HS : 87

3. Tipe data awal:
Tahun                      str
Kode_Reporter            int64
Negara_Reporter            str
Kode_Mitra               int64
Negara_Mitra               str
Kode_Arus                  str
Arus_Perdagangan           str
Kode_HS                    str
Nama_Barang                str
Satuan_Kuantitas           str
Kuantitas              float64
Berat_Bersih_Kg        float64
Nilai_Impor_USD        float64
Dilaporkan_Langsung       bool
dtype: object


In [14]:
def bersihkan_teks(teks):
    if pd.isna(teks) or not str(teks).strip():
        return pd.NA
    return " ".join(str(teks).split())


def ubah_numerik(seri, isi_kosong=0):
    return pd.to_numeric(seri, errors="coerce").fillna(isi_kosong)


def bersihkan_data_impor(df):
    hasil = df.copy()

    for kolom in ["Negara_Reporter", "Negara_Mitra", "Arus_Perdagangan",
                  "Nama_Barang", "Satuan_Kuantitas"]:
        if kolom in hasil.columns:
            hasil[kolom] = hasil[kolom].apply(bersihkan_teks)

    hasil = hasil.dropna(subset=["Kode_HS", "Nama_Barang"])
    hasil["Tahun"] = pd.to_numeric(hasil["Tahun"], errors="coerce").astype("Int64")
    hasil["Kode_HS"] = hasil["Kode_HS"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)

    for kolom in ["Kuantitas", "Berat_Bersih_Kg", "Nilai_Impor_USD"]:
        if kolom in hasil.columns:
            hasil[kolom] = ubah_numerik(hasil[kolom])

    if "Satuan_Kuantitas" in hasil.columns:
        hasil["Satuan_Kuantitas"] = hasil["Satuan_Kuantitas"].fillna("N/A")
    if "Dilaporkan_Langsung" in hasil.columns:
        hasil["Dilaporkan_Langsung"] = hasil["Dilaporkan_Langsung"].astype("boolean")

    hasil = hasil[
        (hasil["Kode_Reporter"] == 360)
        & (hasil["Kode_Mitra"] == 156)
        & (hasil["Kode_Arus"] == "M")
    ]
    hasil = hasil.drop_duplicates(subset=["Tahun", "Kode_HS"], keep="first")
    hasil = hasil.sort_values("Nilai_Impor_USD", ascending=False).reset_index(drop=True)
    return hasil


df_bersih = bersihkan_data_impor(df_impor)
df_bersih.head()

,Tahun,Kode_Reporter,Negara_Reporter,Kode_Mitra,Negara_Mitra,Kode_Arus,Arus_Perdagangan,Kode_HS,Nama_Barang,Satuan_Kuantitas,Kuantitas,Berat_Bersih_Kg,Nilai_Impor_USD,Dilaporkan_Langsung
0,2024,360,Indonesia,156,China,M,Import,890190,"Vessels; n.e.c. in heading no. 8901, for the t...",N/A,0.0,309793014.0,402301919.0,True
1,2024,360,Indonesia,156,China,M,Import,851761,Base stations,N/A,0.0,2061896.0,218958692.0,True
2,2024,360,Indonesia,156,China,M,Import,999999,Commodities not specified according to kind,N/A,0.0,0.0,201725107.0,False
3,2024,360,Indonesia,156,China,M,Import,846599,"Machine-tools; for working wood, cork, bone, h...",N/A,0.0,40700734.0,197091079.0,True
4,2024,360,Indonesia,156,China,M,Import,840681,"Turbines; steam and other vapour turbines, (fo...",N/A,0.0,25930933.0,192492206.0,True


In [15]:
# Pemeriksaan akhir / quality gate
assert len(df_bersih) >= 100, "Dataset akhir belum mencapai minimal 100 baris."
assert df_bersih["Kode_HS"].notna().all(), "Masih ada Kode HS kosong."
assert df_bersih["Nama_Barang"].notna().all(), "Masih ada nama barang kosong."
assert not df_bersih.duplicated(subset=["Tahun", "Kode_HS"]).any(), "Masih ada duplikat."
assert (df_bersih["Kode_Reporter"] == 360).all()
assert (df_bersih["Kode_Mitra"] == 156).all()
assert (df_bersih["Kode_Arus"] == "M").all()

print(f"Jumlah baris akhir       : {len(df_bersih)}")
print(f"Minimal 100 baris        : {len(df_bersih) >= 100}")
print(f"Total nilai kosong       : {int(df_bersih.isna().sum().sum())}")
print(f"Jumlah duplikat          : {df_bersih.duplicated(subset=['Tahun', 'Kode_HS']).sum()}")
print("\nTipe data akhir:")
print(df_bersih.dtypes)

Jumlah baris akhir       : 413
Minimal 100 baris        : True
Total nilai kosong       : 0
Jumlah duplikat          : 0

Tipe data akhir:
Tahun                    Int64
Kode_Reporter            int64
Negara_Reporter            str
Kode_Mitra               int64
Negara_Mitra               str
Kode_Arus                  str
Arus_Perdagangan           str
Kode_HS                    str
Nama_Barang                str
Satuan_Kuantitas           str
Kuantitas              float64
Berat_Bersih_Kg        float64
Nilai_Impor_USD        float64
Dilaporkan_Langsung    boolean
dtype: object


In [16]:
top_10 = (
    df_bersih[["Kode_HS", "Nama_Barang", "Berat_Bersih_Kg", "Nilai_Impor_USD"]]
    .nlargest(10, "Nilai_Impor_USD")
)
top_10

,Kode_HS,Nama_Barang,Berat_Bersih_Kg,Nilai_Impor_USD
0,890190,"Vessels; n.e.c. in heading no. 8901, for the t...",309793014.0,402301919.0
1,851761,Base stations,2061896.0,218958692.0
2,999999,Commodities not specified according to kind,0.0,201725107.0
3,846599,"Machine-tools; for working wood, cork, bone, h...",40700734.0,197091079.0
4,840681,"Turbines; steam and other vapour turbines, (fo...",25930933.0,192492206.0
5,870410,"Vehicles; dumpers, designed for off-highway us...",36583857.0,165316948.0
6,847420,"Machines; for crushing or grinding earth, ston...",35086227.0,124473304.0
7,847780,Machinery; for working rubber or plastics or f...,20127641.0,120849706.0
8,842833,"Elevators and conveyors; continuous-action, fo...",25062560.0,101793404.0
9,842619,"Cranes; transporter, gantry and bridge cranes",24507668.0,99460633.0


In [17]:
total_nilai = df_bersih["Nilai_Impor_USD"].sum()
total_berat = df_bersih["Berat_Bersih_Kg"].sum()

print(f"Total nilai impor pada record yang diambil : USD {total_nilai:,.2f}")
print(f"Total berat bersih                      : {total_berat:,.2f} kg")
print(f"Komoditas bernilai impor terbesar       : {top_10.iloc[0]['Nama_Barang']}")

Total nilai impor pada record yang diambil : USD 3,275,943,128.00
Total berat bersih                      : 905,982,963.00 kg
Komoditas bernilai impor terbesar       : Vessels; n.e.c. in heading no. 8901, for the transport of goods and other vessels for the transport of both persons and goods


In [18]:
nama_file = "dataset_impor_indonesia_dari_china_2024.csv"
df_bersih.to_csv(nama_file, index=False, encoding="utf-8-sig")

df_cek = pd.read_csv(nama_file, dtype={"Kode_HS": str})
print(f"Data disimpan: {nama_file}")
print(f"Verifikasi: {len(df_cek)} baris dan {len(df_cek.columns)} kolom")
df_cek.head()

Data disimpan: dataset_impor_indonesia_dari_china_2024.csv
Verifikasi: 413 baris dan 14 kolom


,Tahun,Kode_Reporter,Negara_Reporter,Kode_Mitra,Negara_Mitra,Kode_Arus,Arus_Perdagangan,Kode_HS,Nama_Barang,Satuan_Kuantitas,Kuantitas,Berat_Bersih_Kg,Nilai_Impor_USD,Dilaporkan_Langsung
0,2024,360,Indonesia,156,China,M,Import,890190,"Vessels; n.e.c. in heading no. 8901, for the t...",NaN,0.0,309793014.0,402301919.0,True
1,2024,360,Indonesia,156,China,M,Import,851761,Base stations,NaN,0.0,2061896.0,218958692.0,True
2,2024,360,Indonesia,156,China,M,Import,999999,Commodities not specified according to kind,NaN,0.0,0.0,201725107.0,False
3,2024,360,Indonesia,156,China,M,Import,846599,"Machine-tools; for working wood, cork, bone, h...",NaN,0.0,40700734.0,197091079.0,True
4,2024,360,Indonesia,156,China,M,Import,840681,"Turbines; steam and other vapour turbines, (fo...",NaN,0.0,25930933.0,192492206.0,True


In [19]:
jumlah_duplikat_mentah = df_impor.duplicated(subset=["Tahun", "Kode_HS"]).sum()
jumlah_kosong_mentah = int(df_impor.isna().sum().sum())

print("=" * 75)
print("RINGKASAN MINI PROJECT")
print("=" * 75)
print("Sumber              : UN Comtrade API")
print("Topik               : Impor barang Indonesia dari China")
print("Periode             : 2024")
print("Klasifikasi         : HS 6 digit")
print(f"Baris mentah        : {len(df_mentah)}")
print(f"Baris akhir         : {len(df_bersih)}")
print(f"Sel kosong awal     : {jumlah_kosong_mentah}")
print(f"Duplikat awal       : {jumlah_duplikat_mentah}")
print(f"Total nilai impor   : USD {total_nilai:,.2f}")
print("Class               : KlienComtrade")
print("Function            : bersihkan_teks, ubah_numerik, bersihkan_data_impor")
print("Output              : dataset_impor_indonesia_dari_china_2024.csv")
print("=" * 75)

RINGKASAN MINI PROJECT
Sumber              : UN Comtrade API
Topik               : Impor barang Indonesia dari China
Periode             : 2024
Klasifikasi         : HS 6 digit
Baris mentah        : 500
Baris akhir         : 413
Sel kosong awal     : 9
Duplikat awal       : 87
Total nilai impor   : USD 3,275,943,128.00
Class               : KlienComtrade
Function            : bersihkan_teks, ubah_numerik, bersihkan_data_impor
Output              : dataset_impor_indonesia_dari_china_2024.csv
